In [0]:
%sql
CREATE OR REPLACE TABLE workspace.silver.ventas AS
WITH dedup AS (
  SELECT *
  FROM workspace.bronze.ventas_raw
  QUALIFY ROW_NUMBER() OVER (
            PARTITION BY pedido_id
            ORDER BY fecha_actualizacion DESC) = 1
)
SELECT
  pedido_id,
  fecha_pedido,
  fecha_actualizacion,
  cliente_id,
  producto_id,
  COALESCE(departamento_destino, 'DESCONOCIDO') AS departamento_destino,
  tm_programadas,
  tm_fabricadas,
  tm_vendidas,
  COALESCE(precio_usd_tm,
           ROUND(AVG(precio_usd_tm) OVER (PARTITION BY producto_id), 2)) AS precio_usd_tm,
  (precio_usd_tm IS NULL) AS es_precio_imputado
FROM dedup;

In [0]:
%sql
SELECT
  COUNT(*)                                                    AS filas,
  COUNT(DISTINCT pedido_id)                                   AS pedidos_unicos,
  SUM(CASE WHEN departamento_destino = 'DESCONOCIDO' THEN 1 ELSE 0 END) AS destinos_desconocidos,
  SUM(CASE WHEN es_precio_imputado THEN 1 ELSE 0 END)         AS precios_imputados
FROM workspace.silver.ventas;